# 실험 02: 멀티턴 대화와 기억력 (Chat History)

## 1. 실험 제목과 목표
- **제목**: 상태 비저장(Stateless) API의 한계 극복 및 대화 문맥 유지 실험
- **목표**: LLM이 이전 대화를 기억하지 못하는 문제를 확인하고, 리스트 기반 방식과 객체 기반(`SimpleMemory`) 방식을 비교 구현해봅니다. 추가로 문맥 길이가 한계에 도달하는 실패 케이스를 체험합니다.

## 2. 실험 개요
1. **기본 예제**: Stateless 특성으로 인한 문맥 단절 확인
2. **비교 실험 1**: 리스트에 History 누적하여 문맥 유지
3. **비교 실험 2**: 대화 도중 System Prompt 삽입/변경 시 모델의 반응 관찰
4. **비교 실험 3**: `SimpleMemory` 객체 클래스를 활용한 구조적 관리
5. **실패 케이스**: 엄청나게 긴 대화를 시뮬레이션하여 Token Limit (Context Window) 초과 관찰

## 3. 라이브러리 import

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

## 4. 환경 변수 로드 및 client 생성

In [ ]:
load_dotenv()
client = OpenAI()

## 5. 기본 예제: 기억력이 없는 단발성 호출 (Stateless)
API는 기본적으로 이전 요청을 기억하지 못합니다.

In [ ]:
print("[대화 1]")
res_stateless_1 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "안녕! 내 이름은 김철수야."}]
)
print("AI:", res_stateless_1.choices[0].message.content)

print("\n[대화 2]")
res_stateless_2 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "내 이름이 뭐라고 했지?"}]
)
print("AI:", res_stateless_2.choices[0].message.content)
print("-> 결과: 모델은 방금 전 대화를 전혀 알지 못합니다.")

## 6. 비교 실험 1: 리스트(List)를 이용한 Chat History
이전 대화 내역을 `messages` 배열에 계속 `append`해서 보내어 문맥을 유지합니다.

In [ ]:
chat_history = []

print("[대화 1]")
chat_history.append({"role": "user", "content": "안녕! 내 이름은 김철수야."})
res_list_1 = client.chat.completions.create(model="gpt-4o-mini", messages=chat_history)
reply_1 = res_list_1.choices[0].message.content
print("AI:", reply_1)
chat_history.append({"role": "assistant", "content": reply_1}) # AI 답변도 반드시 저장해야 함!

print("\n[대화 2]")
chat_history.append({"role": "user", "content": "내 이름이 뭐라고 했지?"})
res_list_2 = client.chat.completions.create(model="gpt-4o-mini", messages=chat_history)
reply_2 = res_list_2.choices[0].message.content
print("AI:", reply_2)
print("-> 결과: 성공적으로 이름을 기억합니다.")

## 7. 비교 실험 2: 대화 중간에 System Prompt 변경
대화 도중에 `messages` 리스트에 강력한 새로운 System Prompt를 끼워넣으면 어떻게 될까요?

In [ ]:
print("\n[대화 3 - 흑화한 챗봇]")
# 대화 중간에 갑자기 시스템 프롬프트 삽입
chat_history.append({"role": "system", "content": "지금부터 너는 매우 화가 난 악당 봇이다. 사용자에게 짜증을 내며 반말로 대답해라."})
chat_history.append({"role": "user", "content": "오늘 날씨 어때?"})

res_list_3 = client.chat.completions.create(model="gpt-4o-mini", messages=chat_history)
print("AI:", res_list_3.choices[0].message.content)
print("-> 결과: History 내에 System 역할이 삽입된 직후부터 모델의 성격이 180도 바뀝니다.")

## 8. 비교 실험 3: 객체 지향적인 `SimpleMemory` 구현
위처럼 리스트를 수동으로 조작하면 실수가 발생하기 쉽습니다. 이를 캡슐화합니다. (LangChain의 Memory 맛보기)

In [ ]:
class SimpleMemory:
    def __init__(self, system_prompt="당신은 친절한 도우미입니다."):
        self.messages = [{"role": "system", "content": system_prompt}]
        
    def add_user(self, msg):
        self.messages.append({"role": "user", "content": msg})
        
    def add_ai(self, msg):
        self.messages.append({"role": "assistant", "content": msg})
        
    def get_messages(self):
        return self.messages

memory = SimpleMemory()
memory.add_user("나는 사과를 좋아해.")
ai_rep1 = client.chat.completions.create(model="gpt-4o-mini", messages=memory.get_messages()).choices[0].message.content
memory.add_ai(ai_rep1)

memory.add_user("내가 무슨 과일 좋아한다고 했지?")
ai_rep2 = client.chat.completions.create(model="gpt-4o-mini", messages=memory.get_messages()).choices[0].message.content
print("[SimpleMemory 결과]:", ai_rep2)

## 9. 실패/주의 케이스: Context Window 초과 시뮬레이션
대화를 무한정 리스트에 담으면 토큰 한도 초과 에러가 발생합니다.

In [ ]:
print("[실패 케이스 - 초거대 데이터 주입]")
huge_memory = SimpleMemory()
huge_text = "가나다라마바사" * 50000  # 엄청난 양의 텍스트

huge_memory.add_user(huge_text)
huge_memory.add_user("안녕?")

try:
    # 의도적으로 에러를 발생시킵니다.
    client.chat.completions.create(model="gpt-4o-mini", messages=huge_memory.get_messages())
except Exception as e:
    print("\n🔥 에러 발생:", type(e).__name__)
    print("에러 메시지:", str(e)[:200] + "...")
    print("\n-> 결과: max_context_length(토큰 한계)를 초과하여 BadRequestError가 발생합니다.")

## 10. 결과 해석

1. **Stateless 보완**: LLM 애플리케이션의 핵심은 '프롬프트 생성기'가 이전 메시지들을 잘 조합해서 모델에게 매번 새롭게 건네주는 데에 있습니다.
2. **메모리의 모듈화**: `list.append` 대신 `Memory` 클래스를 사용하면 관리가 용이해집니다.
3. **Context Window 문제**: 대화가 길어질수록 1회 호출당 발생하는 토큰(비용)이 선형적으로 증가하며, 결국 한계(Limit)를 초과하게 됩니다.

## 11. Notion 실험 로그

### 발견한 문제점
* 챗봇처럼 작동하게 하려면 모든 대화 내역을 계속 보내야 하는데, 이러면 뒤로 갈수록 API 호출 한 번당 소모되는 토큰(비용)이 기하급수적으로 커지고 속도도 느려짐.
* 결국 Context Window 초과 에러(`BadRequestError`)를 피할 수 없음.

### 다음 개선 방향
* 대화 내역이 너무 길어지면 1) 오래된 대화를 자르거나(Window Buffer) 2) 이전 대화를 LLM을 시켜 짧게 요약(Summary Memory)하는 방식을 도입해야 함. (LangChain에서 학습 예정)